[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-07-toy-model.ipynb)

# Lecture 7: your first model, and what it does not know

This is one exercise in three parts.

| part | when | what you get |
|---|---|---|
| **A. In class** | today, twenty minutes | a working install, a trained toy model, a loss curve you can read, and Google Drive connected for Lecture 8a |
| **B. Outside class** | before Lecture 8a | the same model with one phrase held out, two scores that disagree, and a training run that survives a disconnect |
| **C. Turn in** | **before Lecture 8a**, on Learning Suite | three numbers and two sentences |

The point of Part B is the point of Lecture 8a: a model that has memorised its training data can
score perfectly on it and still know nothing. You will see that happen on twelve phrases, and
then in Lecture 8a you will hear why it happens on 100,000.

**Running in Google Colab? Runtime → Change runtime type → GPU first.** This model is small
enough to train on a CPU in under a minute, so a missing GPU will not stop you today. It will in
Lecture 8a, so build the habit now.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 7 (*Research Paper Reviews; Intro to Neural Networks*). A head start on **A7**.

## Part A, in class

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo>=0.2.2"
import torchlingo

In [ ]:
# Check the setup. If this cell raises, stop here and put your hand up.
from torchlingo.colab import setup

env = setup()                  # prints the versions and the device
device = env.device            # a GPU is fine to lack today; not in Lecture 8a

import torch
from pathlib import Path
from torch.utils.data import DataLoader
from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset, collate_fn
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer
from torchlingo.inference import translate_batch
from torchlingo.diagnostics import uniform_loss

### A1. The corpus

Twelve English phrases and their Spanish, each repeated sixteen times so the model has enough
practice to memorise them. Notice the last four: **the cat sleeps, the dog runs, the cat runs,
the dog sleeps** share every word between them. Hold onto that; it is what Part B is about.

In [ ]:
import pandas as pd

BASE_PAIRS = [
    ("Hello world", "Hola mundo"),
    ("Good morning", "Buenos días"),
    ("Good night", "Buenas noches"),
    ("Thank you", "Gracias"),
    ("I love you", "Te amo"),
    ("How are you", "Cómo estás"),
    ("See you tomorrow", "Hasta mañana"),
    ("The cat sleeps", "El gato duerme"),
    ("The dog runs", "El perro corre"),
    ("The cat runs", "El gato corre"),
    ("The dog sleeps", "El perro duerme"),
    ("I love the cat", "Amo al gato"),
]
REPEATS = 16

def write_corpus(pairs, path):
    df = pd.DataFrame(pairs * REPEATS, columns=["src", "tgt"])
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False)
    return df

df = write_corpus(BASE_PAIRS, "data/train_all.tsv")
print(f"{len(df)} training examples from {len(BASE_PAIRS)} distinct phrases.")

### A2. Vocabulary, model, and the number to compare against

Three things the slides told you to look at get printed here: the vocabulary size, the parameter
count, and `ln(V)`, the loss of a model that has learned nothing and guesses every token equally.

In [ ]:
config = Config(
    d_model=64, n_heads=4,
    num_encoder_layers=2, num_decoder_layers=2,
    d_ff=128, dropout=0.1,
    batch_size=8, learning_rate=1e-3,
    num_workers=0,
)

def build(train_tsv, seed=0):
    ds = NMTDataset(train_tsv, config=config)
    loader = DataLoader(ds, batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0)
    torch.manual_seed(seed)
    model = SimpleTransformer(
        src_vocab_size=len(ds.src_vocab), tgt_vocab_size=len(ds.tgt_vocab), config=config,
    ).to(device)
    return ds, loader, model

train_ds, train_loader, model = build("data/train_all.tsv")
V = len(train_ds.tgt_vocab)
n_params = sum(p.numel() for p in model.parameters())
print(f"Source vocabulary: {len(train_ds.src_vocab)} tokens.  Target vocabulary V = {V} tokens.")
print(f"Parameters: {n_params:,}")
print(f"ln(V) = {uniform_loss(V):.3f}   <- a first-batch loss near this means 'not learned anything yet'")

### A3. Train

Forty epochs. It runs in seconds on a GPU and under a minute on a CPU. Watch two things: the
**first** logged loss against `ln(V)` above, and where the curve flattens.

Expect the first logged loss to sit *a little above* `ln(V)`, not below it. A freshly built
model is not perfectly uniform: its random weights already lean towards some tokens, and
leaning the wrong way costs more than not leaning at all. What tells you it is learning is the
curve crossing `ln(V)` and continuing down. Each `log_every` line is the mean of the last five
batches, and `train_losses[0]` is the mean of the whole first epoch, so neither is one batch.

In [ ]:
EPOCHS = 40
result = train_model(
    model, train_loader,
    num_epochs=EPOCHS, device=device, config=config,
    gradient_clip=1.0, log_every=5,
    # Saves progress at every epoch. Running this cell again continues from the save instead of
    # retraining, so a second run finishes at once. Part B shows why that matters.
    checkpointer=TrainingCheckpointer("lecture-07-all", checkpoint_dir="checkpoints/lecture-07-all", verbose=False),
)
losses = result.train_losses
print(f"\nEpoch-1 average loss: {losses[0]:.3f}   final loss: {losses[-1]:.3f}   ln(V): {uniform_loss(V):.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 3.5))
plt.plot(range(1, len(losses) + 1), losses, label="training loss (epoch mean)")
plt.axhline(uniform_loss(V), linestyle="--", label=f"ln(V) = {uniform_loss(V):.2f}")
plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.grid(alpha=0.3); plt.legend()
plt.title("Your first loss curve")
plt.show()

### A4. Translate the training phrases

In [ ]:
def translate(model, ds, sentences):
    return translate_batch(model, sentences, ds.src_vocab, ds.tgt_vocab,
                           decode_strategy="greedy", device=device, config=config)

for src, hyp in zip([p[0] for p in BASE_PAIRS[:6]], translate(model, train_ds, [p[0] for p in BASE_PAIRS[:6]])):
    print(f"{src:20} -> {hyp}")

### A5. Connect Google Drive, today rather than in Lecture 8a

The A8 notebook in Lecture 8a reads your Assignment 5 corpus from Google Drive and saves your
split, your checkpoints and your best model back to it, because a Colab runtime forgets
everything when it disconnects and Drive does not.

The first time a notebook mounts Drive, Google asks you to choose an account and approve access.
It takes a minute. Do it now, while someone can help, rather than in Lecture 8a with a training
run waiting.

The next cell mounts Drive, creates the folder the A8 notebook looks in, `MyDrive/CS479`, and
writes a small file there to prove it can.

In [ ]:
# Mount Google Drive and prove you can write to it. The A8 notebook makes this same call.
env = setup(drive=True)        # in Colab, Google asks you to approve access the first time

COURSE_DIR = env.drive_dir / "CS479"
COURSE_DIR.mkdir(parents=True, exist_ok=True)
check = COURSE_DIR / "lecture-07-drive-check.txt"
check.write_text("Drive works. The A8 notebook reads and writes this folder.\n", encoding="utf-8")

print(f"Wrote {check}")
print()
print("Before Lecture 8a, put your two Assignment 5 files in this folder:")
print(f"  {COURSE_DIR}/")
print("The A8 notebook looks for them there; you will type their names into it.")

**Report back, in class:** your first logged loss and `ln(V)`; your final loss; whether the
translations are right. Also anything that broke, because you are the first class to run this
and the person next to you probably hit it too.

---

## Part B, outside class: hold one phrase out

Everything above scored the model on sentences it was trained on. That is not a test; it is a
memory check. So: remove **"The dog sleeps"** from the training set, train a fresh model on the
other eleven, and ask it to translate the one it never saw.

It has seen *the dog* (in "The dog runs") and *sleeps* (in "The cat sleeps"). Whether it can put
them together is the question. Either answer is the lesson.

In [ ]:
HELD_OUT = ("The dog sleeps", "El perro duerme")
train_pairs = [p for p in BASE_PAIRS if p != HELD_OUT]
assert len(train_pairs) == 11

write_corpus(train_pairs, "data/train_heldout.tsv")
ho_ds, ho_loader, ho_model = build("data/train_heldout.tsv")

# One checkpoint folder per held-out phrase, so changing HELD_OUT trains a new model rather than
# continuing the old one.
ho_name = "lecture-07-without-" + HELD_OUT[0].lower().replace(" ", "-")
ho_result = train_model(ho_model, ho_loader, num_epochs=EPOCHS, device=device, config=config,
                        gradient_clip=1.0, log_every=0,
                        checkpointer=TrainingCheckpointer(ho_name, checkpoint_dir=f"checkpoints/{ho_name}", verbose=False))
print(f"Held-out model: final loss {ho_result.train_losses[-1]:.3f}")

In [ ]:
import sacrebleu

seen_src  = [p[0] for p in train_pairs]
seen_ref  = [p[1] for p in train_pairs]
seen_hyp  = translate(ho_model, ho_ds, seen_src)
unseen_hyp = translate(ho_model, ho_ds, [HELD_OUT[0]])

# Exact match is the honest metric for three-word phrases. chrF works on short text too.
exact_seen = sum(h == r for h, r in zip(seen_hyp, seen_ref))
chrf_seen   = sacrebleu.corpus_chrf(seen_hyp, [seen_ref]).score       # one hypothesis stream, one list of reference streams
chrf_unseen = sacrebleu.corpus_chrf(unseen_hyp, [[HELD_OUT[1]]]).score

print(f"The 11 phrases it trained on:   {exact_seen}/11 exactly right,   chrF {chrf_seen:5.1f}")
print(f"The 1 phrase it never saw:      chrF {chrf_unseen:5.1f}")
print(f"\n  {HELD_OUT[0]!r}  ->  {unseen_hyp[0]!r}     (wanted {HELD_OUT[1]!r})")

# And BLEU, which you met in Lecture 6. Watch what it does here.
bleu_seen   = sacrebleu.corpus_bleu(seen_hyp, [seen_ref]).score
bleu_unseen = sacrebleu.corpus_bleu(unseen_hyp, [[HELD_OUT[1]]]).score
print(f"\nBLEU, seen: {bleu_seen:.1f}   BLEU, unseen: {bleu_unseen:.1f}")

**What to look at.** The seen line should read 11/11 and chrF 100: the model memorised its
training set, and scoring on the training set rewards exactly that. The unseen line is the only
number that says anything about *translation*. If it recombined *el perro* and *duerme*, it
generalised from three related phrases. If it produced something else (a common outcome is
*El perro corre*, the verb it saw next to *perro*), it did not: sixteen repetitions of eleven
phrases is not enough to learn that *sleeps* is *duerme* independently of which animal is doing
it.

Either way you have just seen what Lecture 7's "One More Way a Number Can Mislead You" slide
warned about: **a score measured on training data is not a measurement.** In Lecture 8a you split
100,000 pairs into train, validation and test precisely so that never happens to your real
number.

**And BLEU came out 0.0 on both.** Not because the translations are wrong, since eleven of them
are perfect, but because BLEU needs 4-gram matches and a three-word sentence has no 4-grams. You
saw this in Lecture 6 on ten short sentences. It is the metric's shape, not the model's quality,
and it is why this notebook reports chrF and exact match instead.

**Optional, five minutes:** change `HELD_OUT` to `("Good night", "Buenas noches")` and rerun the
two cells above. *Night* → *noches* appears nowhere else in the corpus, so there is nothing to
recombine from. Compare the two unseen scores and think about what generalisation requires.

### Also before Lecture 8a: watch a training run survive a disconnect

The A8 model trains for hours, and a Colab session can drop in the middle: a closed laptop, an
idle timeout. Both `train_model` calls above passed a **`TrainingCheckpointer`**, which saves
the training state (the weights, the optimizer, how far it got) at the end of every epoch, and
every ten minutes within one. Run a cell with the same checkpointer again and it **continues**
from the last save instead of starting over.

On a real run the only test of that is a real disconnect, which is a bad time to find out. So
here is one on purpose, in seconds. The next cell trains a fresh model, pretends the session
drops partway through epoch 12, then does what you would do after reconnecting: builds the model
again from scratch and runs the same training call. Read the `[checkpoint]` lines.

In [ ]:
import tempfile


class DisconnectsAfter:
    """A stand-in for a Colab session that drops: the loader stops after this many batches."""

    def __init__(self, loader, batches):
        self.loader, self.left = loader, batches

    def __len__(self):
        return len(self.loader)

    def __iter__(self):
        for batch in self.loader:
            if self.left == 0:
                raise ConnectionError("the session dropped")
            self.left -= 1
            yield batch


DEMO_EPOCHS = 20
demo_dir = tempfile.mkdtemp(prefix="resume-demo-")   # a new, empty folder every time you run this

# 1. Start training, with a checkpointer. The "session" drops 5 batches into epoch 12.
_, loader, demo_model = build("data/train_all.tsv")
try:
    train_model(demo_model, DisconnectsAfter(loader, 11 * len(loader) + 5),
                num_epochs=DEMO_EPOCHS, device=device, config=config, gradient_clip=1.0,
                log_every=0, checkpointer=TrainingCheckpointer("resume-demo", checkpoint_dir=demo_dir))
except ConnectionError as dropped:
    print(f"\n*** {dropped}. Everything in memory is gone, including the model. ***\n")

# 2. "Reconnect": build the model again from scratch, then run the same training call.
_, loader, demo_model = build("data/train_all.tsv")
demo = train_model(demo_model, loader,
                   num_epochs=DEMO_EPOCHS, device=device, config=config, gradient_clip=1.0,
                   log_every=0, checkpointer=TrainingCheckpointer("resume-demo", checkpoint_dir=demo_dir))

print(f"\nEpochs in the loss history: {len(demo.train_losses)} of {DEMO_EPOCHS}, "
      f"though the second call trained only the last {DEMO_EPOCHS - 11}.")

**What to look at.** The second call printed `resumed from epoch 10`, and its first line is
epoch 12, not epoch 1. The rebuilt model started from random weights; the checkpointer put the
trained ones back. (Its epochs count from 0, so its epoch 10 is the eleventh.)

Look at what was lost: the five batches of epoch 12 that ran before the drop. The last save was
the end of epoch 11, so epoch 12 ran again from its start. That is the whole cost of a
disconnect when you checkpoint: at most the work since the last save. Without a checkpointer it
is everything.

In Lecture 8a the same two things happen with no tricks. The checkpoints go to Google Drive
rather than a temporary folder, because a Colab runtime forgets its own disk when it drops, and
"running the same call again" means rerunning the training cell.

---

## Part C, turn in

Paste the output of the next cell into the Assignment 7 box on Learning Suite, then answer the
two questions under it in your own words. **Due before Lecture 8a.** This is graded for
completion and for whether the two sentences engage with the numbers, not for the numbers.

In [ ]:
print("=== Lecture 7 toy model: report ===")
print(f"Target vocabulary V = {V};  ln(V) = {uniform_loss(V):.3f}")
print(f"Final training loss, all 12 phrases:          {losses[-1]:.3f}")
print(f"Held-out run, the 11 training phrases:  {exact_seen}/11 exact,  chrF {chrf_seen:.1f}")
print(f"Held-out run, the 1 unseen phrase:      chrF {chrf_unseen:.1f}")
print(f"Held-out translation: {HELD_OUT[0]!r} -> {unseen_hyp[0]!r}   (wanted {HELD_OUT[1]!r})")
print(f"BLEU on both: {bleu_seen:.1f} and {bleu_unseen:.1f}")
print()
print("1. What does the gap between the seen and unseen scores tell you about what the model learned?")
print("2. Assignment 8 trains on 100,000 pairs. Name one thing you will do differently because of this exercise.")

Lecture 8a opens with "how did the toy model go" and then "why a trained model comes out bad".
Bring your held-out result; the second slide is about it.